# DeepSeek-V4-Flash-Vision-Exp on 2x DGX Spark — SGLang (preview build) |  |  | | --- | --- | | | **Verdict: FAIL-TO-BOOT.** The SGLang cookbook's verified DGX Spark cell (`flash-vision / fp4 / balanced / multi-2`) does not complete boot on this node pair. Four independent failure classes were hit and documented with receipts; the deepest progress (all weights + DSpark draft loaded, 78 GB/rank) deadlocks in verify-CUDA-graph capture with both GPUs idle. |  | | | | Metric | Value | Source | | |---|---|---|---| | | Decode, c=1 | n/a — boot blocked | this notebook | | | Best aggregate throughput | n/a — boot blocked | this notebook | | | Prefill | n/a — boot blocked | this notebook | | | TTFT | n/a — boot blocked | this notebook | |

## DeepSeek-V4-Flash-Vision-Exp — 2x DGX Spark, SGLang TP=2 (preview `dev-v4f-2dgx-v2`) |  | Executed notebook. Every cell output below is committed; see `notebooks/README.md` for the section order and the `LIVE` flag convention. **This notebook documents a failed boot**: it is published because the failure classes are reproducible, receipted, and actionable upstream.

In [ ]:
# --- Status cell -------------------------------------------------------
# LIVE = False: this notebook replays the committed receipts under
# results/2026-09-04-deepseek-v4-flash-vision-exp-2node-tp2-sglang/.
# There is no running endpoint to point at: the stack never finished booting.
LIVE = False
RESULTS_DIR = "../results/2026-09-04-deepseek-v4-flash-vision-exp-2node-tp2-sglang"
print("LIVE =", LIVE)

## 1. TL;DR |  | **Verdict: FAIL-TO-BOOT** on 2x DGX Spark (GB10 x2, ConnectX-7 RoCE), SGLang preview image `lmsysorg/sglang:dev-v4f-2dgx-v2` (manifest `sha256:67873eb9…`), checkpoint `deepseek-ai/DeepSeek-V4-Flash-Vision-Exp` @ `6821d6ad` (tip of main — the cookbook cell pins no revision). Ten boots were executed against the verbatim cell and four progressively-corrected variants. |  | Four failure classes, all receipted: |  | | # | Class | Stage | Blocker | |---|---|---|---| | 1 | compatibility | rendezvous | Gloo connectFullMesh refused — worker announced `127.0.0.1` (host `127.0.1.1 hostname` convention) | | 2 | capacity | weight loading | kernel OOM-killer: loader staging ~39 GB anon on top of ~84 GB/rank weights (121.7/119.6 GiB usable unified) | | 3 | capacity | memory pool | runtime computes minimum viable `mem-fraction-static = 0.8244`; cookbook pins `0.80` | | 4 | stability | verify graph capture | deadlock 30+ min, both GPUs 0% / 13 W on both ranks, no log progress |  | Classes 1-3 have fixes (host wiring, mmap loader + swap, 0.83). Class 4 is unreached-by-fix and unreconciled: with every fix applied, capture hangs. |  | **Corroboration**: containers `dsv4-vision-sglang-head` / `dsv4-vision-sglang-worker` (Exited 1, 2 days before this experiment, older preview image `dev-dsv4-flash-vision`) on the same nodes indicate an earlier, independent attempt hit the same wall. |  | **Actionable upstream**: the image tag `dev-v4f-2dgx-v2` was silently repushed between two pulls on the same day (33.3 GB → 48.8 GB build). Pin by digest, and re-verify the published cell on hardware with ≤121.7 GiB usable: the cell's own minimum-viable mem-fraction (0.8244) exceeds its published pin (0.80) on us.

In [ ]:
import json, os
with open(os.path.join(RESULTS_DIR, 'summary.json')) as f:
    summary = json.load(f)
print('verdict:', summary['verdict'])
print('failure classes:', len(summary['failure_classes']))
for fc in summary['failure_classes']:
    print(f"  [{fc['class']}] {fc['where']}: {fc['error'][:90]}...")
print()
print('pins.image_note:', summary['pins']['image_note'][:120], '...')

## 2. Visible results |  | There are no throughput results: the stack never served a request. The visible artifacts are the failure receipts. |  | ### 2.1 Failure class 2 receipt — kernel OOM during weight loading |  | Kernel ring (read via privileged container on each node; hosts keep no user-accessible dmesg):

In [ ]:
for node in ['apollo1', 'apollo2']:
    path = os.path.join(RESULTS_DIR, f'receipt-kernel-oom-{node}.txt')
    print(f'--- {path}')
    print(open(path).read())

### 2.2 Failure class 3 receipt — runtime minimum-viable mem-fraction |  | Verbatim runtime error on the verbatim `0.80` boot: |  | ``` | ValueError: Loaded weights leave no GPU memory for the KV cache under | --mem-fraction-static=0.8. Raise --mem-fraction-static above 0.825 | (minimum viable = 1 - available/pre = 0.8244). If using speculative | decoding, draft weights are now counted. | ``` |  | ### 2.3 Failure class 4 receipt — verify-graph capture deadlock |  | Last log lines on both ranks before the lane was stopped: |  | ``` | [TP0] Capture target verify CUDA graph begin. backend=full, |   num_tokens_per_req=6, bs=[1,2,3,4,5,6,7,8,10,12,14,16,18,20,22,24,26,28,30,32], |   avail mem=29.41 GB          # 10:00:07, then 30+ min silence | [TP1] Using DeepseekV4AttnBackend for dsv4 attention backend (CUDA).   # 10:00:08, then silence | ``` |  | `nvidia-smi` sampled during the silence: both GPUs `0 %`, `13.2 W / 13.7 W` (idle).

## 3. Reproduce |  | ### Hardware | 2x DGX Spark (GB10, 121.7 / 119.6 GiB usable unified), ConnectX-7 RoCE direct (head `10.100.120.2`, worker `10.100.120.1`). |  | ### Verbatim cookbook cell (fails at class 1 → 2 → 3 in sequence) |  | ```bash | docker run --gpus all --shm-size 32g --network host \
  --ulimit memlock=-1:-1 --cap-add IPC_LOCK --device /dev/infiniband \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  --env SGLANG_SM120_FLASHMLA_BACKEND=b12x --env B12X_MLA_SM120_DSV4_H16_NATIVE=1 \
  --env SGLANG_OPT_FUSE_MHC_POST_PRE=1 --env SGLANG_OPT_FP8_WO_A_GEMM=1 \
  --env SGLANG_SKIP_SGL_KERNEL_VERSION_CHECK=1 --env SGLANG_B12X_MAX_TOKENS=8192 \
  --env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  lmsysorg/sglang:dev-v4f-2dgx-v2 \
  sglang serve --trust-remote-code \
    --model-path deepseek-ai/DeepSeek-V4-Flash-Vision-Exp \
    --tp 2 --nnodes 2 --node-rank 0 --dist-init-addr 10.100.120.2:20000 \
    --moe-runner-backend b12x --speculative-algorithm DSPARK \
    --chunked-prefill-size 8192 --context-length 327680 \
    --mem-fraction-static 0.80 --swa-full-tokens-ratio 0.2 \
    --cuda-graph-max-bs-decode 32 --max-running-requests 32 \
    --host 0.0.0.0 --port 30000
# rank 1: same command, --node-rank 1
``` |  | Expected on this node pair: class-1 Gloo refusal, then (after fixing host wiring) class-2 loader OOM, then class-3 ValueError. |  | ### Deepest-progress variant (classes 1-3 fixed; hangs at class 4) |  | Same command plus: |  | ```bash
  --model-loader-extra-config '{"enable_multithread_load": false}' \
  --mem-fraction-static 0.83 \
``` |  | and, on the host (outside docker): |  | ```text
/etc/hosts: map each node's hostname to its ConnectX-7 IP (replace the
            127.0.0.1 <hostname> Ubuntu default line)
64 GB swapfile per node, swapon
``` |  | Boot timeline of the deepest run: checkpoint shards 48/48 verified on both nodes → rank 0 `Load weight end` 409 s (75.50 GB) → DSpark draft 3.24 GB → verify capture begins 10:00:07 → deadlock. |  | ### Pointing at a different endpoint |  | Not applicable — no endpoint ever came up.

<details>
<summary>## 4. Appendix</summary>

### Boot ledger

| # | Config delta vs cookbook | Outcome |
|---|---|---|
| 1 | verbatim | class 1 (Gloo refused) |
| 2 | `GLOO_SOCKET_IFNAME`/`NCCL_SOCKET_IFNAME` added | class 2 (loader OOM, rank 0 `-9`) |
| 3 | roles swapped (rank 0 on the other node) | class 2 again — not node-specific |
| 4 | `num_threads: 2` in loader extra-config | class 2 again (staging still ~4 shards) |
| 5 | `num_threads: 1` | class 2 again — staging is the scheduler's own 3-shard window (39.3 GB anon in dmesg) |
| 6 | `enable_multithread_load: false` (mmap) + swap on one node | rank 0 loaded 429 s; rank 1 OOM'd — swap was missing there |
| 7 | mmap + swap on both, `mem-fraction 0.72` | class 3 (pool leaves no KV room) |
| 8 | `mem-fraction 0.80` restored | class 3 — runtime names minimum viable 0.8244 |
| 9 | `mem-fraction 0.83` | weights + draft loaded; class 4 capture deadlock |
| 10 | rerun with host wiring as permanent fix | same class 4 |

### Image double-push gotcha

`lmsysorg/sglang:dev-v4f-2dgx-v2` resolved to two different builds within one day (33.3 GB image id `7e333ce…`, then 48.8 GB `67873eb…`); both pulls reported the same tag. The club's digest-pinning rule exists for exactly this.

### Safety and cost

No data exposure: the only secrets on the nodes are the vLLM API key (unchanged, cluster-internal) and HF tokens (not used — checkpoint is ungated). Two human-hours and ~15 node-hours, mostly download and boot retries. The 164 GB NVFP4 text-checkpoint download from a parallel lane and the 64 GB swapfiles remain on the nodes (`/host/swapfile`); swap is left on deliberately for the next preview-build attempt.

### Upstream links

- Cookbook cell: https://docs.sglang.io/cookbook/autoregressive/DeepSeek/DeepSeek-V4
- Image: `lmsysorg/sglang:dev-v4f-2dgx-v2` (branch `b12x-vision` @ `452239a74f` per cookbook notes)
- Evidence: `results/2026-09-04-deepseek-v4-flash-vision-exp-2node-tp2-sglang/` in this repository
</details>